# LightGBM — pilot thời gian trên 10 chuỗi M5

Chạy tuần tự trên Google Colab CPU. Mã nguồn và cấu hình được lấy từ nhánh `thuytien`.
Dữ liệu gốc giữ riêng trong runtime; chỉ kết quả nhỏ được lưu để báo cáo.
Train d_1–d_1885, dự báo 28 ngày, 3 lần đo; không tuning, không dùng nhãn Validation/Test.
CSV Drive thiếu khóa d/id được kiểm tra và chuẩn hóa trong bản đầu vào riêng, không đổi doanh số.
Đây là pilot thời gian, không phải lần chạy Validation chính thức M04 hoặc kiểm tra đủ 5 CSV S06.


In [ ]:
from pathlib import Path
import json, sys, subprocess, hashlib, platform, os, zipfile, datetime
REPO = Path('/content/IS403.R11-M5-Competition')
if not REPO.exists():
    subprocess.run(['git','clone','--branch','thuytien','--single-branch',
                    'https://github.com/haiphamt/IS403.R11-M5-Competition.git',str(REPO)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','lightgbm==4.6.0','numpy==2.4.6',
               'pandas==2.2.3','scikit-learn==1.6.1','PyYAML==6.0.3','matplotlib>=3.8,<4','gdown>=5,<6'],check=True)
subprocess.run([sys.executable,'-m','unittest','discover','-s','tests',
                '-p','test_lightgbm*.py','-v'],cwd=REPO,check=True)


In [ ]:
import gdown, pandas as pd
RAW = REPO/'data/raw'
RAW.mkdir(parents=True, exist_ok=True)
INPUTS = {
 'calendar.csv': '1rAO0QS5HPoUprlzw5mGll572yqnChPjU',
 'sales_train_validation.csv': '1QGNDfsD7rxzCCbEY3-Y1mcbLcgKbofw4',
 'sell_prices.csv': '1_-7xIYafyMG45E157DJh6sgGvOHswaO1',
}
EXPECTED = {'calendar.csv':(1969,14),'sales_train_validation.csv':(30490,1919),'sell_prices.csv':(6841121,4)}
checks = []
for name, file_id in INPUTS.items():
    path = RAW/name
    # Download atomically so an interrupted runtime cannot reuse a partial CSV.
    temp = path.with_suffix('.download')
    gdown.download(id=file_id, output=str(temp), quiet=False)
    temp.replace(path)
    rows = 0
    for chunk in pd.read_csv(path, chunksize=100000):
        rows += len(chunk)
        ncols = len(chunk.columns)
    columns = set(pd.read_csv(path,nrows=0).columns)
    if name == 'calendar.csv':
        # Original M5 Drive calendar lacks Kaggle's synthetic day key d.
        required = {'date','wm_yr_wk','wday','month','year','event_name_1','event_type_1','event_name_2','event_type_2','snap_CA','snap_TX','snap_WI'}
        assert rows == 1969 and ncols in (13,14) and required <= columns, (name, rows, ncols, sorted(columns))
    elif name == 'sales_train_validation.csv':
        assert rows == 30490 and ncols in (1918,1919), (name,rows,ncols)
        assert {'item_id','dept_id','cat_id','store_id','state_id'} | {f'd_{i}' for i in range(1,1914)} <= columns
    else:
        assert (rows,ncols) == EXPECTED[name], (name, rows, ncols)
    h = hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda: f.read(1024*1024), b''):
            h.update(block)
    checks.append(dict(filename=name, rows=rows, columns=ncols, sha256=h.hexdigest(), source='https://drive.google.com/file/d/'+file_id+'/view'))
ENV = dict(task='C03', owner='Thủy Tiên', recorded_at=datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=7))).isoformat(),
           runtime='colab', python=platform.python_version(), logical_cpu_count=os.cpu_count(),
           ram=subprocess.check_output(['free','-h'],text=True), nproc=subprocess.check_output(['nproc'],text=True).strip(),
           input_checks=checks, s06_scope='3/5 required CSVs verified; NOT a completed S06 evidence package')
MODEL_RAW = REPO/'data/pilot_input'
MODEL_RAW.mkdir(parents=True,exist_ok=True)
cal = pd.read_csv(RAW/'calendar.csv')
dates = pd.to_datetime(cal['date'],errors='raise')
assert dates.tolist() == pd.date_range('2011-01-29',periods=1969).tolist()
derived = 'd_' + ((dates-pd.Timestamp('2011-01-29')).dt.days+1).astype(str)
if 'd' in cal:
    assert cal['d'].equals(derived)
else:
    cal['d'] = derived
cal.to_csv(MODEL_RAW/'calendar.csv',index=False)
sales = pd.read_csv(RAW/'sales_train_validation.csv')
derived_ids = sales['item_id'] + '_' + sales['store_id'] + '_validation'
assert derived_ids.nunique() == 30490
if 'id' in sales:
    assert sales['id'].equals(derived_ids)
else:
    sales.insert(0,'id',derived_ids)
sales.to_csv(MODEL_RAW/'sales_train_validation.csv',index=False)
del sales
target = MODEL_RAW/'sell_prices.csv'
if not target.exists():
    os.link(RAW/'sell_prices.csv',target)
ENV['calendar_transformation'] = 'Original calendar.csv retained; verified 1969 consecutive dates starting 2011-01-29; derived d_1..d_1969 from date into separate pilot_input/calendar.csv. No sales changed.'
ENV['sales_id_transformation'] = 'Original sales CSV retained; derived unique id=item_id+store_id+validation suffix in separate pilot_input file; no sales values changed.'
ENV['pilot_calendar_sha256'] = hashlib.sha256((MODEL_RAW/'calendar.csv').read_bytes()).hexdigest()
print(json.dumps(ENV,ensure_ascii=False,indent=2))


In [ ]:
OUTPUT=REPO/'results/tables'
before=set(OUTPUT.glob('c03_lightgbm_*'))
subprocess.run([sys.executable,'scripts/benchmark_lightgbm.py','--data-dir',str(MODEL_RAW),'--repeats','3'],cwd=REPO,check=True)
created=set(OUTPUT.glob('c03_lightgbm_*'))-before
assert len(created)==1
RUN=created.pop()
summary=json.loads((RUN/'summary.json').read_text())
timings=pd.read_csv(RUN/'runtime_repeats.csv')
means=pd.read_csv(RUN/'runtime_by_series.csv')
pred=pd.read_csv(RUN/'validation_forecasts.csv')
import numpy as np
assert len(timings)==30 and len(means)==10 and len(pred)==280
assert pred.series_id.nunique()==10 and np.isfinite(pred.prediction).all() and (pred.prediction>=0).all()
assert summary['runtime']=='colab' and not summary['validation_labels_used'] and not summary['test_labels_used']
(RUN/'environment_and_input_checks.json').write_text(json.dumps(ENV,ensure_ascii=False,indent=2),encoding='utf-8')
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(10,5))
ax.barh(means.series_id,means.mean_total_seconds)
ax.set_xlabel('Mean fit + predict seconds (3 repeats, CPU)')
ax.set_title('LightGBM pilot - 10 original M5 series')
fig.tight_layout()
FIG=REPO/'results/figures'/summary['run_id']
FIG.mkdir(parents=True,exist_ok=True)
fig.savefig(FIG/'runtime_by_series.png',dpi=160)
plt.show()
display(means[['series_id','mean_fit_seconds','mean_predict_seconds','mean_total_seconds']])
print('Run:',summary['run_id'])
print('Seconds per series:',summary['mean_fit_predict_seconds_per_series'])
print('1200-series extrapolation (not measured full sample):',summary['estimated_sample_fit_predict_seconds'])
ZIP=Path('/content')/(summary['run_id']+'.zip')
with zipfile.ZipFile(ZIP,'w',zipfile.ZIP_DEFLATED) as z:
    for folder in [RUN,FIG]:
        for p in folder.iterdir():
            z.write(p,p.relative_to(REPO))
from google.colab import files
files.download(str(ZIP))
